In [1]:
import numpy as np 
import pandas as pd 
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
from rouge_score import rouge_scorer
from datasets import Dataset

In [15]:
df = pd.read_csv('train.csv')

In [16]:
df = df.iloc[:10000]

In [18]:
df.head()

,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...


In [19]:
df.drop(columns=['id'] , inplace=True) 

C:\Users\FHCS\AppData\Local\Temp\ipykernel_20300\2345858219.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['id'] , inplace=True)


In [20]:
def preprocess(txt):
    txt = txt.replace('/n' , ' ')
    txt = ' '.join(txt.split())
    return txt
df['article'] = df['article'].apply(preprocess)
df['highlights'] = df['highlights'].apply(preprocess)

C:\Users\FHCS\AppData\Local\Temp\ipykernel_20300\2306047351.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['article'] = df['article'].apply(preprocess)
C:\Users\FHCS\AppData\Local\Temp\ipykernel_20300\2306047351.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['highlights'] = df['highlights'].apply(preprocess)


In [21]:
df['article'] = "summarize: " + df['article']
df

C:\Users\FHCS\AppData\Local\Temp\ipykernel_20300\1847741688.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['article'] = "summarize: " + df['article']


,article,highlights
0,summarize: By . Associated Press . PUBLISHED: ...,"Bishop John Folda, of North Dakota, is taking ..."
1,summarize: (CNN) -- Ralph Mata was an internal...,Criminal complaint: Cop used his role to help ...
2,summarize: A drunk driver who killed a young w...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,summarize: (CNN) -- With a breezy sweep of his...,Nina dos Santos says Europe must be ready to a...
4,summarize: Fleetwood are the only team still t...,Fleetwood top of League One after 2-0 win at S...
...,...,...
9995,summarize: A renowned poet was murdered by her...,Police found writer's body dead after family r...
9996,summarize: (Health.com) -- A class of injectab...,The research contradicts numerous earlier stud...
9997,summarize: By . Rob Cooper . A mother who acci...,"Nicola Millar, 36, was unable to pay the fee a..."
9998,summarize: They are hated by many as an eyesor...,Google is developing machines which will be te...


In [22]:
dataset = Dataset.from_pandas(df)

In [23]:
tokenzier = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [24]:
device = torch.device('cpu')
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
max_input_length = 256   # shorter for CPU
max_target_length = 64  
def tokenize_batch(batch):
    inputs = tokenzier(batch['article'] , max_length=max_input_length , truncation=True , padding='max_length')
    targets = tokenzier(batch['highlights'] , max_length=max_target_length , truncation=True , padding='max_length')
    batch['input_ids'] = inputs['input_ids']
    batch['attention_mask'] = inputs['attention_mask']
    batch['labels'] = targets['input_ids']
    return batch
tokenized_dataset = dataset.map(tokenize_batch , batched=True , remove_columns=['article' , 'highlights'])
tokenized_dataset.set_format(type=torch ,columns= ['input_ids','attention_mask','labels'])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

ValueError: Format type should be one of [None, 'arrow', 'numpy', 'pandas', 'custom', 'torch', 'tensorflow'], but got '<module 'torch' from 'C:\\Users\\FHCS\\anaconda3\\Lib\\site-packages\\torch\\__init__.py'>'

In [26]:
train_size = int(0.9 * len(tokenized_dataset))  
train_dataset = tokenized_dataset.select(range(train_size))
eval_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

In [30]:
training_args = TrainingArguments(
    output_dir="./t5_cnn_cpu",
    per_device_train_batch_size=1,  # CPU -> keep 1
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=500,
    eval_strategy="steps",
    save_total_limit=2,
    fp16=False,  # CPU cannot use mixed precision
    report_to="none",  # disable wandb/logging
) 

In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [ ]:
trainer.train()

C:\Users\FHCS\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
50,3.445105,3.002890
100,2.650320,2.336390
150,2.350392,2.098642
200,2.224960,2.057545
250,2.448181,2.047468


In [ ]:
def generate_summary(text):
    text = "summarize: " + preprocess_text(text)
    inputs = tokenizer.encode(text, return_tensors="pt", max_length=max_input_length, truncation=True)
    outputs = model.generate(inputs, max_length=max_target_length, num_beams=2, early_stopping=True)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary 